# IBNR Estimation — Four Actuarial Methods

This notebook estimates **Incurred But Not Reported (IBNR)** reserves using four standard actuarial methods applied to a paid loss triangle.

**Methods covered:**
1. Chain Ladder (Development Method)
2. Average Development Method
3. Bornhuetter-Ferguson Method
4. Cape Cod Method

**Data:** 10 accident years (2015–2024) × 10 development periods (12–120 months)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

## 1. Load Data

In [ ]:
# Load paid loss triangle
triangle_raw = pd.read_csv('../data/paid_loss_triangle.csv', index_col='accident_year')
triangle_raw.columns = [int(c.replace('dev_', '')) for c in triangle_raw.columns]

# Load earned premium
premium = pd.read_csv('../data/earned_premium.csv', index_col='accident_year')['earned_premium']

print('Paid Loss Triangle ($000s):')
triangle_raw.style.format('{:,.0f}', na_rep='').set_caption('Paid Losses by Accident Year and Development Month')

In [ ]:
# Convert to numpy array for calculations
AYs = triangle_raw.index.tolist()       # accident years
devs = triangle_raw.columns.tolist()    # development periods
n = len(AYs)
triangle = triangle_raw.values.astype(float)  # shape (n, n)

# Visualise the triangle as a heatmap
fig, ax = plt.subplots(figsize=(12, 5))
mask = np.isnan(triangle)
data_display = np.where(mask, np.nan, triangle)

im = ax.imshow(data_display, aspect='auto', cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='Paid Losses ($000s)')

ax.set_xticks(range(n))
ax.set_xticklabels([f'{d}m' for d in devs], rotation=45)
ax.set_yticks(range(n))
ax.set_yticklabels(AYs)
ax.set_xlabel('Development Period')
ax.set_ylabel('Accident Year')
ax.set_title('Paid Loss Triangle Heatmap')

for i in range(n):
    for j in range(n):
        if not np.isnan(triangle[i, j]):
            ax.text(j, i, f'{triangle[i,j]:,.0f}', ha='center', va='center', fontsize=7)

plt.tight_layout()
plt.show()

## 2. Compute Age-to-Age Development Factors

In [ ]:
def compute_ata_factors(triangle):
    """Compute age-to-age (link ratio) factors for each development column.
    
    Uses the volume-weighted average (sum of col j+1 / sum of col j)
    over all rows that have data in both columns.
    """
    n_cols = triangle.shape[1]
    ata = np.full(n_cols - 1, np.nan)
    
    for j in range(n_cols - 1):
        col_curr = triangle[:, j]
        col_next = triangle[:, j + 1]
        mask = ~np.isnan(col_curr) & ~np.isnan(col_next)
        if mask.sum() > 0:
            ata[j] = col_next[mask].sum() / col_curr[mask].sum()
    
    return ata


def compute_simple_avg_ata(triangle):
    """Compute simple average (unweighted) of individual link ratios per column."""
    n_cols = triangle.shape[1]
    ata = np.full(n_cols - 1, np.nan)
    
    for j in range(n_cols - 1):
        col_curr = triangle[:, j]
        col_next = triangle[:, j + 1]
        mask = ~np.isnan(col_curr) & ~np.isnan(col_next)
        if mask.sum() > 0:
            ratios = col_next[mask] / col_curr[mask]
            ata[j] = ratios.mean()
    
    return ata


# Volume-weighted ATA factors
ata_vw = compute_ata_factors(triangle)
# Simple average ATA factors
ata_sa = compute_simple_avg_ata(triangle)

# Build individual link ratio table for display
link_ratios = pd.DataFrame(index=AYs, columns=[f'{devs[j]}-{devs[j+1]}' for j in range(n-1)], dtype=float)
for i in range(n):
    for j in range(n - 1):
        if not np.isnan(triangle[i, j]) and not np.isnan(triangle[i, j+1]):
            link_ratios.iloc[i, j] = triangle[i, j+1] / triangle[i, j]

link_ratios.loc['Vol. Wtd. Avg'] = ata_vw
link_ratios.loc['Simple Avg'] = ata_sa

print('Individual Link Ratios:')
link_ratios.style.format('{:.4f}', na_rep='').highlight_min(axis=0, color='#d4e8ff').highlight_max(axis=0, color='#ffd4d4')

In [ ]:
# Compute cumulative development factors (CDF) from tail to each period
def cdfs_from_ata(ata, tail=1.0):
    """Compute CDFs (ultimate development factors) from age-to-age factors.
    
    CDF[j] = product of all ATA factors from period j to ultimate.
    """
    factors = list(ata) + [tail]
    cdfs = np.ones(len(factors))
    cdfs[-1] = tail
    for i in range(len(factors) - 2, -1, -1):
        cdfs[i] = factors[i] * cdfs[i + 1]
    return cdfs  # one per column (len = n_cols)

TAIL = 1.0  # assume fully developed at 120 months
cdfs_vw = cdfs_from_ata(ata_vw, tail=TAIL)
cdfs_sa = cdfs_from_ata(ata_sa, tail=TAIL)

cdf_df = pd.DataFrame({
    'Development': [f'{d}m' for d in devs],
    'CDF (Vol. Wtd.)': cdfs_vw,
    'CDF (Simple Avg)': cdfs_sa,
    '% Developed (Vol. Wtd.)': 1 / cdfs_vw * 100
})
print('Cumulative Development Factors:')
cdf_df.set_index('Development').style.format({
    'CDF (Vol. Wtd.)': '{:.4f}',
    'CDF (Simple Avg)': '{:.4f}',
    '% Developed (Vol. Wtd.)': '{:.1f}%'
})

## 3. Method 1: Chain Ladder

Projects the latest diagonal forward using volume-weighted age-to-age factors:

$$\hat{U}_i = C_{i,d_i} \times CDF_{d_i}$$

where $C_{i,d_i}$ is the latest known cumulative paid loss for accident year $i$ and $CDF_{d_i}$ is the cumulative development factor from the current age to ultimate.

In [ ]:
def chain_ladder(triangle, cdfs):
    """Chain Ladder method.
    
    Returns:
        ultimates : array of ultimate loss estimates per accident year
        latest    : latest diagonal values
        ibnr      : IBNR = ultimate - latest
    """
    n_rows, n_cols = triangle.shape
    latest = np.full(n_rows, np.nan)
    latest_col = np.full(n_rows, 0, dtype=int)
    
    for i in range(n_rows):
        row = triangle[i]
        known = np.where(~np.isnan(row))[0]
        if len(known) > 0:
            j = known[-1]
            latest[i] = row[j]
            latest_col[i] = j
    
    ultimates = latest * cdfs[latest_col]
    ibnr = ultimates - latest
    return ultimates, latest, ibnr, latest_col


ult_cl, latest, ibnr_cl, latest_col = chain_ladder(triangle, cdfs_vw)

cl_df = pd.DataFrame({
    'Accident Year': AYs,
    'Latest Paid': latest,
    'CDF': cdfs_vw[latest_col],
    'Ultimate (Chain Ladder)': ult_cl,
    'IBNR': ibnr_cl
}).set_index('Accident Year')

totals = cl_df.sum()
totals.name = 'TOTAL'
cl_df = pd.concat([cl_df, totals.to_frame().T])

print('Chain Ladder Results ($000s):')
cl_df.style.format({
    'Latest Paid': '{:,.0f}',
    'CDF': '{:.4f}',
    'Ultimate (Chain Ladder)': '{:,.0f}',
    'IBNR': '{:,.0f}'
}).apply(lambda x: ['font-weight: bold' if x.name == 'TOTAL' else '' for _ in x], axis=1)

## 4. Method 2: Average Development Method

Same structure as Chain Ladder but uses **simple (unweighted) average** link ratios instead of volume-weighted averages. Gives equal weight to each accident year's experience regardless of volume.

In [ ]:
ult_ad, _, ibnr_ad, _ = chain_ladder(triangle, cdfs_sa)

ad_df = pd.DataFrame({
    'Accident Year': AYs,
    'Latest Paid': latest,
    'CDF (Simple Avg)': cdfs_sa[latest_col],
    'Ultimate (Avg Dev)': ult_ad,
    'IBNR': ibnr_ad
}).set_index('Accident Year')

totals = ad_df.sum()
totals.name = 'TOTAL'
ad_df = pd.concat([ad_df, totals.to_frame().T])

print('Average Development Results ($000s):')
ad_df.style.format({
    'Latest Paid': '{:,.0f}',
    'CDF (Simple Avg)': '{:.4f}',
    'Ultimate (Avg Dev)': '{:,.0f}',
    'IBNR': '{:,.0f}'
}).apply(lambda x: ['font-weight: bold' if x.name == 'TOTAL' else '' for _ in x], axis=1)

## 5. Method 3: Bornhuetter-Ferguson (BF)

Blends the Chain Ladder ultimate with an *a priori* expected loss estimate:

$$\hat{U}_i^{BF} = C_{i,d_i} + \left(1 - \frac{1}{CDF_{d_i}}\right) \times ELR \times P_i$$

where $ELR$ is the expected loss ratio and $P_i$ is earned premium for accident year $i$. The term $(1 - 1/CDF)$ is the expected **unreported fraction**.

In [ ]:
def bornhuetter_ferguson(latest, latest_col, cdfs, premium, elr):
    """Bornhuetter-Ferguson method.
    
    Args:
        elr     : expected loss ratio (a priori)
        premium : earned premium per accident year
    """
    unreported_pct = 1 - 1 / cdfs[latest_col]  # % yet to be reported
    expected_losses = elr * premium
    ibnr = unreported_pct * expected_losses
    ultimates = latest + ibnr
    return ultimates, ibnr, unreported_pct, expected_losses


# Derive ELR from Chain Ladder ultimates (a common practical approach)
# ELR = sum(CL ultimates) / sum(premiums)
premium_arr = premium.loc[AYs].values
ELR = ult_cl.sum() / premium_arr.sum()
print(f'A priori ELR (derived from Chain Ladder): {ELR:.1%}')

ult_bf, ibnr_bf, unrep_pct, exp_losses = bornhuetter_ferguson(
    latest, latest_col, cdfs_vw, premium_arr, ELR
)

bf_df = pd.DataFrame({
    'Accident Year': AYs,
    'Latest Paid': latest,
    'Unreported %': unrep_pct * 100,
    'Expected Losses': exp_losses,
    'IBNR': ibnr_bf,
    'Ultimate (BF)': ult_bf
}).set_index('Accident Year')

totals = bf_df.agg({'Latest Paid': 'sum', 'Expected Losses': 'sum', 'IBNR': 'sum', 'Ultimate (BF)': 'sum', 'Unreported %': 'mean'})
totals.name = 'TOTAL'
bf_df = pd.concat([bf_df, totals.to_frame().T])

print('\nBornhuetter-Ferguson Results ($000s):')
bf_df.style.format({
    'Latest Paid': '{:,.0f}',
    'Unreported %': '{:.1f}%',
    'Expected Losses': '{:,.0f}',
    'IBNR': '{:,.0f}',
    'Ultimate (BF)': '{:,.0f}'
}).apply(lambda x: ['font-weight: bold' if x.name == 'TOTAL' else '' for _ in x], axis=1)

## 6. Method 4: Cape Cod

Like BF, but the ELR is **estimated from the data** rather than assumed:

$$ELR^{CC} = \frac{\sum_i C_{i,d_i}}{\sum_i P_i / CDF_{d_i}}$$

The denominator $P_i / CDF_{d_i}$ is the **used-up premium** — the portion of premium that has been "exposed" to reported losses.

In [ ]:
def cape_cod(latest, latest_col, cdfs, premium):
    """Cape Cod method — ELR estimated from data."""
    used_up_premium = premium / cdfs[latest_col]  # P_i * (% developed)
    elr_cc = latest.sum() / used_up_premium.sum()
    
    unreported_pct = 1 - 1 / cdfs[latest_col]
    ibnr = unreported_pct * elr_cc * premium
    ultimates = latest + ibnr
    return ultimates, ibnr, elr_cc, used_up_premium, unreported_pct


ult_cc, ibnr_cc, elr_cc, used_up_prem, unrep_pct_cc = cape_cod(
    latest, latest_col, cdfs_vw, premium_arr
)

print(f'Cape Cod ELR (derived from data): {elr_cc:.1%}')

cc_df = pd.DataFrame({
    'Accident Year': AYs,
    'Latest Paid': latest,
    'Used-Up Premium': used_up_prem,
    'Unreported %': unrep_pct_cc * 100,
    'IBNR': ibnr_cc,
    'Ultimate (Cape Cod)': ult_cc
}).set_index('Accident Year')

totals = cc_df.agg({'Latest Paid': 'sum', 'Used-Up Premium': 'sum', 'IBNR': 'sum', 'Ultimate (Cape Cod)': 'sum', 'Unreported %': 'mean'})
totals.name = 'TOTAL'
cc_df = pd.concat([cc_df, totals.to_frame().T])

print('\nCape Cod Results ($000s):')
cc_df.style.format({
    'Latest Paid': '{:,.0f}',
    'Used-Up Premium': '{:,.0f}',
    'Unreported %': '{:.1f}%',
    'IBNR': '{:,.0f}',
    'Ultimate (Cape Cod)': '{:,.0f}'
}).apply(lambda x: ['font-weight: bold' if x.name == 'TOTAL' else '' for _ in x], axis=1)

## 7. Results Comparison

In [ ]:
comparison = pd.DataFrame({
    'Accident Year': AYs,
    'Latest Paid': latest,
    'IBNR — Chain Ladder': ibnr_cl,
    'IBNR — Avg Development': ibnr_ad,
    'IBNR — Bornhuetter-Ferguson': ibnr_bf,
    'IBNR — Cape Cod': ibnr_cc,
}).set_index('Accident Year')

totals = comparison.sum()
totals.name = 'TOTAL'
comparison_display = pd.concat([comparison, totals.to_frame().T])

print('IBNR Comparison by Method ($000s):')
comparison_display.style.format('{:,.0f}').apply(
    lambda x: ['font-weight: bold' if x.name == 'TOTAL' else '' for _ in x], axis=1
).background_gradient(subset=[
    'IBNR — Chain Ladder', 'IBNR — Avg Development',
    'IBNR — Bornhuetter-Ferguson', 'IBNR — Cape Cod'
], cmap='YlOrRd', axis=None)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: IBNR by accident year, stacked comparison ---
ax = axes[0]
x = np.arange(len(AYs))
width = 0.2
methods = {
    'Chain Ladder': ibnr_cl,
    'Avg Development': ibnr_ad,
    'Bornhuetter-Ferguson': ibnr_bf,
    'Cape Cod': ibnr_cc
}
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']

for k, (label, values) in enumerate(methods.items()):
    ax.bar(x + k * width, values, width, label=label, color=colors[k], alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(AYs, rotation=45)
ax.set_ylabel('IBNR ($000s)')
ax.set_title('IBNR by Accident Year and Method')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# --- Right: Total IBNR by method ---
ax2 = axes[1]
totals_ibnr = [v.sum() for v in methods.values()]
bars = ax2.bar(list(methods.keys()), totals_ibnr, color=colors, alpha=0.85)
ax2.set_ylabel('Total IBNR ($000s)')
ax2.set_title('Total IBNR by Method')
ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))

for bar, val in zip(bars, totals_ibnr):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
             f'{val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('ibnr_comparison.png', bbox_inches='tight')
plt.show()
print('Chart saved to ibnr_comparison.png')

In [ ]:
# Development pattern chart
fig, ax = plt.subplots(figsize=(10, 5))

for i, ay in enumerate(AYs):
    row = triangle[i]
    known_idx = np.where(~np.isnan(row))[0]
    known_devs = [devs[j] for j in known_idx]
    known_vals = row[known_idx]
    ax.plot(known_devs, known_vals / known_vals[-1], marker='o', markersize=4,
            label=str(ay), alpha=0.7)

ax.set_xlabel('Development Period (months)')
ax.set_ylabel('% of Latest Diagonal')
ax.set_title('Development Pattern by Accident Year (normalised to latest diagonal)')
ax.legend(fontsize=8, loc='upper left', ncol=2)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

## 8. Summary Statistics

In [ ]:
summary = pd.DataFrame({
    'Method': list(methods.keys()),
    'Total IBNR ($000s)': [v.sum() for v in methods.values()],
    'Total Ultimate ($000s)': [latest.sum() + v.sum() for v in methods.values()],
    'Total Premium ($000s)': [premium_arr.sum()] * 4,
})

summary['Ultimate Loss Ratio'] = summary['Total Ultimate ($000s)'] / summary['Total Premium ($000s)']
summary['IBNR / Latest Paid'] = [v.sum() / latest.sum() for v in methods.values()]
summary = summary.set_index('Method')

print('Summary:')
summary.style.format({
    'Total IBNR ($000s)': '{:,.0f}',
    'Total Ultimate ($000s)': '{:,.0f}',
    'Total Premium ($000s)': '{:,.0f}',
    'Ultimate Loss Ratio': '{:.1%}',
    'IBNR / Latest Paid': '{:.1%}'
})

## Method Notes

| Method | Strengths | Weaknesses |
|---|---|---|
| **Chain Ladder** | Simple, data-driven, widely used | Amplifies distortions in early diagonals; poor for immature years |
| **Average Development** | Smooths volume extremes; equal year weight | May underweight credible large-volume years |
| **Bornhuetter-Ferguson** | Stabilises immature years with prior; blends well | Sensitive to choice of ELR |
| **Cape Cod** | ELR derived from data; no external assumption | Can be distorted if loss experience is volatile |